# Paquetes de Python

In [2]:
import numpy as np
from typing import Dict

# 2. Linear Layer Backward

In [2]:
def linear_backward(dout: np.ndarray, x: np.ndarray, w: np.ndarray, b: np.ndarray) -> Dict[str, np.ndarray]:
    """
    Computes dx, dw, db for y = x @ w + b.
    
    Args:
        dout: Upstream gradient (N, Dout)
        x: Input (N, Din)
        w: Weights (Din, Dout)
        b: Bias (Dout,)
        
    Returns:
        Dict with "dx", "dw", "db"
    """
    dx =  dout @ w.T
    dw = x.T @ dout
    db = dout.sum(axis=0)
    return {"dx":dx, "dw":dw, "db":db}
    pass

## Contexto

En una capa lineal de una red neuronal, la operación directa se define como:

$$
y = x \cdot W + b
$$

donde:
- $x \in \mathbb{R}^{N \times D_{in}}$: matriz de entrada
- $W \in \mathbb{R}^{D_{in} \times D_{out}}$: matriz de pesos
- $b \in \mathbb{R}^{D_{out}}$: vector de sesgos
- $y \in \mathbb{R}^{N \times D_{out}}$: salida de la capa

Durante el **backpropagation**, se busca calcular los gradientes de la función de pérdida L respecto a cada parámetro: x, W y b.

---

## Derivadas parciales

### 1️. Gradiente respecto a la entrada x
Aplicando la regla de la cadena:

$$
\frac{\partial L}{\partial x} = \frac{\partial L}{\partial y} \cdot \frac{\partial y}{\partial x}
$$

Como $\dfrac{\partial y}{\partial x} = W$, se obtiene:

$$
dx = dout \cdot W^T
$$

donde `dout` representa el gradiente que proviene de la capa siguiente $\frac{\partial L}{\partial y}$.

En código:
```python
dx =  dout @ w.T
```

---

### 2️. Gradiente respecto a los pesos W
De forma análoga:

$$
\frac{\partial L}{\partial W} = x^T \cdot \frac{\partial L}{\partial y}
$$

por lo tanto:

$$
dw = x^T \cdot dout
$$

En código:
```python
dw = x.T @ dout
```

---

### 3️. Gradiente respecto al sesgo b
El sesgo se suma a cada fila de la salida, por lo que su gradiente es la suma de los gradientes de salida a lo largo del batch:

$$
db = \sum_{i=1}^{N} dout_i
$$

En código:  
```python
db = dout.sum(axis=0)
```

# 6. Tanh Activation

In [3]:
def tanh_ops(x: np.ndarray, dout: np.ndarray) -> Dict[str, np.ndarray]:
    """
    Computes tanh forward and backward.
    """
    out = np.tanh(x)

    tanh_prime = 1 - np.tanh(x)**2
    dx = dout*tanh_prime

    return {"out":out, "dx":dx}
    pass

## Contexto

La función de activación tangente hiperbólica se define como:

$$
\tanh(x) = \frac{\sinh(x)}{\cosh(x)} = \frac{e^x - e^{-x}}{e^x + e^{-x}}
$$

Su rango es \(-1, 1\), y se utiliza para normalizar valores en redes neuronales.

---

## Derivación del gradiente
Para el **backward pass**, necesitamos la derivada de $\tanh(x)$ respecto a x:

$$
\frac{d}{dx}\tanh(x) = 1 - \tanh^2(x)
$$

Esto se obtiene aplicando la regla del cociente o usando la identidad hiperbólica:

$$
\cosh^2(x) - \sinh^2(x) = 1
$$

Por tanto:

$$
\frac{d}{dx}\tanh(x) = \frac{\cosh^2(x) - \sinh^2(x)}{\cosh^2(x)} = 1 - \tanh^2(x)
$$

---

## Interpretación en el backward pass
Durante la propagación hacia atrás, el gradiente de la pérdida L respecto a la entrada x se calcula como:

$$
dx = dout \cdot (1 - \tanh^2(x))
$$

donde `dout` es el gradiente que proviene de la capa siguiente.

# 10. SGD Optimizer

In [4]:
def sgd_step(w: np.ndarray, dw: np.ndarray, lr: float) -> np.ndarray:
    """
    Updates w using SGD.
    """
    step = lr*dw
    w_new = w - step

    return w_new
    pass

## Contexto

El **Descenso de Gradiente Estocástico (SGD)** es el algoritmo de optimización más básico y fundamental para entrenar redes neuronales.  
Su objetivo es minimizar una función de pérdida L(w) ajustando los parámetros w en la dirección opuesta al gradiente.

---

## Derivación matemática

La actualización de los parámetros se define como:

$$
w_{\text{new}} = w - \eta \frac{\partial L}{\partial w}
$$

donde:
- w: vector de pesos actuales  
- $\eta$: tasa de aprendizaje (learning rate)  
- $\frac{\partial L}{\partial w}$: gradiente de la pérdida respecto a los pesos  

En el caso estocástico, el gradiente se calcula sobre un **subconjunto (batch)** de los datos, lo que introduce cierta variabilidad pero acelera el entrenamiento.

---

## Interpretación geométrica
El gradiente $\frac{\partial L}{\partial w}$ apunta hacia la dirección de **mayor aumento** de la pérdida.  
Por tanto, el paso de actualización $ -\eta \frac{\partial L}{\partial w} $ mueve los parámetros **en sentido contrario**, buscando el mínimo local.

$$
\text{SGD step} = -\eta \cdot \text{gradient}
$$

Cada iteración reduce la pérdida L(w) hasta converger a un valor mínimo.